        tf version of Vineet's code
        added support for biorthogonal wavelets
        removed numpy dependency
        removed feqz dependency
        vectorized the forward and inverse transforms

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-16 21:24:41.909367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750089281.930825  871243 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750089281.937264  871243 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-16 21:24:41.959692: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import tensorflow as tf
import numpy as np
import pywt
# from scipy.signal import freqz
import matplotlib.pyplot as plt
import time


class ShearletTransform2D(tf.keras.layers.Layer):
    """Shearlet transform 2D"""
    def __init__(
        self, N, 
        J, 
        L=None, B=None, 
        a=None, 
        norm=False, 
        wave='db3',
        transform=None, ## inverse
        **kwargs):
        super(ShearletTransform2D, self).__init__(**kwargs)
       
        self.transform = transform ## inverse
        self.wave = wave
        self.π = tf.constant(np.pi, dtype=tf.float64)
        self.N = N
        self.J = J
        self.L = L
        self.B = B
        self.norm = norm
        self.shearlet_system = self.get_shearlet_system('wavedec')
        if 'bio' in self.wave:
            self.shearlet_system_rec = self.get_shearlet_system('waverec')
        else: self.shearlet_system_rec = None
        if a is not None:
            L = [int(tf.math.round(a**((1-1/2)*j))) for j in range(J)]
            B = [a**(j+1)/L[j] for j in range(J)]
        if self.norm:
            self.norm_factors = self.get_norm_factors()
        
    @tf.function
    def get_norm_factors(self):
        """ Normalizing with thre reconstruction Filterbank!!! recheck here!!!"""       
        FBdec, _ = self.getFB()
        # Compute the squared L2 norm over the spatial dimensions
        norm_factors = tf.reduce_sum(tf.math.square(tf.math.abs(FBdec)), axis=(-2, -1), keepdims=True)  # shape: [F, 1, 1, 1]
        norm_factors = tf.math.sqrt(self.N**2/norm_factors)
        return tf.cast(norm_factors, tf.complex128)
         
    @tf.function
    def v(self, x):
        x = tf.cast(x, tf.float64)  # Ensure consistent dtype if needed
        # Constants casted to match x's dtype
        ten = tf.constant(10.0, dtype=x.dtype)
        fifteen = tf.constant(15.0, dtype=x.dtype)
        six = tf.constant(6.0, dtype=x.dtype)
        one = tf.constant(1.0, dtype=x.dtype)
        zero = tf.constant(0.0, dtype=x.dtype)
        # Conditions
        cond0 = x <= zero
        cond1 = x >= one
        # Polynomial part for 0 < x < 1
        x_poly = x**3 * (ten - fifteen * x + six * x**2)
        return tf.where(cond0, zero, tf.where(cond1, one, x_poly))
    
    @tf.function
    def phi(self, x):
        x = tf.convert_to_tensor(x, dtype=tf.float64)
        x = tf.abs(x) / self.π#tf.constant(np.pi, dtype=tf.float64)
        # Conditions
        cond1 = x <= 1
        cond2 = tf.logical_and(x > 1, x < 2)
        # phi parts
        part1 = tf.ones_like(x)
        part2 = tf.cos(0.5 * self.π * self.v(x - 1))
        part3 = tf.zeros_like(x)

        return tf.where(cond1, part1, tf.where(cond2, part2, part3))
    
    @tf.function
    def W(self, x, ljbj, ljbj_1):
        return tf.math.sqrt(tf.math.square(self.phi(x/ljbj))-tf.math.square(self.phi(x/ljbj_1)))

    
    @tf.function
    def _V_piecewise_fn(self, x):
        # x = tf.convert_to_tensor(x, dtype=tf.float32)
        cond = tf.logical_and(x > -1, x < 1)
        val = tf.sqrt(self.v(1.0 - tf.abs(x)))
        return tf.where(cond, val, tf.zeros_like(x))
    @tf.function
    def _V_wavedec(self, x):
        wavelet = pywt.Wavelet(self.wave)
        h0 = wavelet.dec_lo
        return self.freqz_abs(x, h0)
    @tf.function
    def _V_waverec(self, x):
        wavelet = pywt.Wavelet(self.wave)
        g0 = wavelet.rec_lo
        return self.freqz_abs(x, g0)
    @tf.function
    def freqz_abs(self, x1, h0):
        # Ensure high precision: float64 and complex128
        # h0 = tf.convert_to_tensor(h0, dtype=tf.complex128)
        x1 = self.π*x1
        
        h0 = h0 / tf.sqrt(tf.constant(2.0, dtype=tf.float64))
        h0 = tf.cast(h0, tf.complex128)

        x1 = tf.cast(x1, tf.float64)
        # π = tf.constant(np.pi, dtype=tf.float64)
        x1_clipped = tf.clip_by_value(x1, -self.π, self.π)
        # x1_clipped = tf.clip_by_value(x1, -tf.constant(tf.constant(np.pi, dtype=tf.float64)), tf.constant(np.pi, dtype=tf.float64))
        freqs = tf.reshape(x1_clipped, [-1])  # [M]

        n = tf.range(tf.shape(h0)[0], dtype=tf.float64)  # [L]
        n = tf.reshape(n, [1, -1])                      # [1, L]
        freqs = tf.reshape(freqs, [-1, 1])              # [M, 1]

        exponent = tf.exp(-1j * tf.cast(freqs * n, tf.complex128))  # [M, L]
        response = tf.matmul(exponent, tf.reshape(h0, [-1, 1]))     # [M, 1]
        response_abs = tf.abs(response)                             # [M, 1]

        return tf.cast(tf.reshape(response_abs, tf.shape(x1)), dtype=tf.float64)
        
    
    @tf.function
    def centered_wrap(self, x, half_modulo=np.pi):
        return (x+half_modulo)%(2*half_modulo)-half_modulo

    # ## How to use this??
    @tf.function
    def downsample(self, x, fac=2):
        return tf.add_n(tf.split(tf.add_n(tf.split(x, fac, axis=-1)), fac, axis=-2))#.numpy()

    # @tf.function
    def get_shearlet_system(self, filtertype):

        def V(x):
            if filtertype == "piecewise":
                return self._V_piecewise_fn(x)
            elif filtertype == "wavedec":
                return self._V_wavedec(x)
            elif filtertype == "waverec":
                return self._V_waverec(x)
            else:
                raise ValueError(f"Unknown mode: {filtertype}")

        N = self.N
        J = self.J
        L = self.L
        B = self.B
        shearlet_system = [[], [], []]
        temp = self.centered_wrap((tf.range(N, dtype=tf.float64) * (2.0 * self.π / N)), self.π)
        w = tf.stack(tf.meshgrid(temp, temp, indexing='ij'), axis=0)
        
        temp = tf.math.abs(w)
        XP0 = temp[1]<=temp[0]
        XP1 = tf.logical_not(XP0)
        XP0 = tf.cast(XP0, dtype=tf.float64)
        XP1 = tf.cast(XP1, dtype=tf.float64)

        w0 = w[0, :, :1]
        shearlet = self.phi((L[J-1]*B[J-1])*w0)
        shearlet = shearlet*XP0 + tf.transpose(shearlet, perm=[1, 0])*XP1
        shearlet_system[0].append(shearlet)
        
        temp = w[0]
        # ratio = np.divide(w[1], temp, out=np.full_like(temp, np.inf), where=temp!=0)
        safe_temp = tf.where(temp != 0, temp, tf.constant(1.0, dtype=temp.dtype))
        ratio = tf.where(temp != 0, w[1] / safe_temp, tf.constant(float('inf'), dtype=temp.dtype))
        # safe_temp = tf.where(temp != 0, temp, tf.constant(np.inf, dtype=temp.dtype))
        # ratio = w[1] / safe_temp

        for j in range(J):
            W0 = self.W(w0, (L[j]*B[j])/(L[J-1]*B[J-1]), (1 if j==0 else L[j-1]*B[j-1])/(L[J-1]*B[J-1]))
            W1 = tf.transpose(W0, perm=[1, 0])*XP1
            W0 = W0*XP0
            temp = L[j]*ratio

            for l1 in range(-L[j], L[j]+1):
                shearlet = V(temp-l1)
                if abs(l1)==L[j]:
                    shearlet = shearlet*W0 + tf.transpose(shearlet, perm=[1, 0])*W1
                    shearlet_system[2].append(shearlet)
                else:
                    shearlet = shearlet*W0
                    shearlet_system[1].append(shearlet)
        for i in range(3):
            shearlet_system[i] = tf.stack(shearlet_system[i], axis=0)
        return shearlet_system

    # UPDATE
    def getFB(self):
        if 'bio' in self.wave:
            return self._analysis_bank_tensor(), self._synthesis_bank_tensor_biortho()
        else:
            return self._analysis_bank_tensor(), None
    def _analysis_bank_tensor(self):
        FB = [
            self.shearlet_system[0],
            self.shearlet_system[1],
            tf.transpose(self.shearlet_system[1], perm=[0, 2, 1]),
            self.shearlet_system[2],
           ]
        return tf.cast(tf.concat(FB, axis=0), tf.complex128)
    def _synthesis_bank_tensor_biortho(self):
        FB = [
            self.shearlet_system_rec[0],
            self.shearlet_system_rec[1],
            tf.transpose(self.shearlet_system_rec[1], perm=[0, 2, 1]),
            self.shearlet_system_rec[2],
        ]
        return tf.cast(tf.concat(FB, axis=0), tf.complex128)

    def forward(self, x):
        x = tf.cast(x, dtype=tf.complex128)
        xfft = tf.signal.fft2d(x)
        # FB = self._analysis_bank_tensor()
        FB, _ = self.getFB()
        filtered = tf.einsum('ij,fij->fij', xfft, FB)        
        filtered = tf.signal.ifft2d(filtered)
        # filtered = filtered[..., :x.shape[-3], :x.shape[-2], :x.shape[-1]]
        if self.norm:
            filtered *= self.norm_factors
        return filtered

    def call(self, x):
        if self.transform==None:
            return self.forward(x)
        elif self.transform=='inverse':
            return self.inverse(x)
        else:
            raise ValueError(f"Unknown key {transform}!! keys 'DST' or 'IDST' only allowed")

    def inverse(self, y):
        y = tf.cast(y, tf.complex128)
        if self.norm:
            y = y/self.norm_factors
        yfft = tf.signal.fft2d(y)             
        if 'bio' in self.wave:
            _, FB = self.getFB()
        else:
            FB, _ = self.getFB()        
        synthesized = tf.einsum('fij,fij->ij', yfft, FB)
        synthesized = tf.signal.ifft2d(synthesized)#, axes=(-3, -2, -1))
        return tf.math.real(synthesized)
        # return synthesized


n = 128
start_time = time.time()
# ST2D = ShearletTransform2D(N=n, J=3, L=[2, 4, 8], B=[1, 1, 1], norm=True)
ST2D = ShearletTransform2D(N=n, J=3, L=[2, 4, 8], B=[1, 1, 1], norm=True, wave='db25')
ST2D = ShearletTransform2D(N=n, J=3, L=[2, 4, 8], B=[1, 1, 1], norm=True, wave='rbio1.5')

print(time.time()-start_time)

x = np.random.randn(n, n)
start_time = time.time()
y = ST2D.forward(x)
print(time.time()-start_time)
print(y.shape)

start_time = time.time()
x_ = ST2D.inverse(y)
print(time.time()-start_time)
print(x_.shape)

print(np.max(np.abs(x_-x)))

2025-06-16 21:24:47.846198: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-06-16 21:24:47.846243: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:137] retrieving CUDA diagnostic information for host: meherangarh
2025-06-16 21:24:47.846250: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:144] hostname: meherangarh
2025-06-16 21:24:47.846348: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:168] libcuda reported version is: 570.148.8
2025-06-16 21:24:47.846373: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:172] kernel reported version is: 570.148.8
2025-06-16 21:24:47.846379: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:259] kernel version seems to match DSO: 570.148.8


1.2003958225250244
0.0225675106048584
(57, 128, 128)
0.014966249465942383
(128, 128)
1.887379141862766e-15


# Batched multichannel Shearlet Transform layer

        Inherits ShearletTransform2D class

In [3]:
class DST2D(ShearletTransform2D):
    def __init__(self, transform=None, **kwargs):#, l1=0.0, l2=0.0):
        super(DST2D, self).__init__(**kwargs)
        self.transform = transform
        if self.transform == 'inverse':
            self.name = 'IDST3D'

    
    ## UPDATE applied for batched multichannel inputs
    def forward(self, x):
        x = tf.cast(x, dtype=tf.complex128)
        # FB = self._yield_filter_bank_tensor()
        FB, _ = self.getFB()
        x = tf.transpose(x, perm=[0,3,1,2])
        xfft = tf.signal.fft2d(x)
        filtered = tf.einsum('bcij,fij->bcfij', xfft, FB)
        filtered = tf.signal.ifft2d(filtered)
        if self.norm:
            # self.norm_factors = tf.cast(self.norm_factors, dtype=tf.complex128)
            filtered *= self.norm_factors
            # filtered = tf.einsum('bcfijk,fijk->bcfijk', filtered, self.norm_factors)
        return filtered
    
    def call(self, x):
        if self.transform==None:
            return self.forward(x)
        elif self.transform=='inverse':
            return self.inverse(x)
        else:
            raise ValueError(f"Unknown key {transform}!! keys 'None' (default) or 'inverse' only allowed")

    def inverse(self, y):
        y = tf.cast(y, tf.complex128)
        self.norm_factors = tf.cast(self.norm_factors, tf.complex128)
        if self.norm:
            y = y/self.norm_factors
            # y = tf.einsum('bcfijk,fijk->bcfijk', y, 1/self.norm_factors)      
        yfft = tf.signal.fft2d(y)
        if 'bio' in self.wave:
            _, FB = self.getFB()
        else:
            FB, _ = self.getFB()  
        synthesized = tf.einsum('bcfij,fij->bcij', yfft, FB)
        synthesized = tf.cast(synthesized, tf.complex128)
        synthesized = tf.signal.ifft2d(synthesized)#, axes=(-3, -2, -1))
        synthesized = tf.transpose(synthesized, perm=[0,2,3,1])
        return tf.math.real(synthesized)


if __name__=='__main__':
    # start_time = time.time()
    # ST3D = ShearletTransform3D(N=128, J=2, L=[1, 2], B=[4, 8], norm=True)
    # ST3D = DST3D(N=32, J=2, L=[1, 2], B=[4, 8], norm=True, wave='db6')
    # ST3D = DST3D(N=32, J=2, L=[1, 2], B=[4, 8], norm=True, wave='db10')
    # ST3D = DST3D(N=128, J=2, L=[1, 2], B=[4, 8], norm=True, wave='bior1.5')

    # ST3D = ShearletTransform3D(N=32, J=3, L=[1, 4, 8], B=[6, 8, 10], norm=True, wave='db20')
    # print(time.time()-start_time)
    
    ## Example 1: model summary
    import numpy as np
    n = 64
    axis1 = np.arange(0,n)
    x = np.einsum('i,j->ij', axis1, axis1)
    # x = np.einsum('i,j,k->ijk', axis1, axis1, axis1)
    xx = tf.expand_dims(tf.expand_dims(x, axis=-1), axis=0)
    xx = tf.concat([xx,xx, xx], axis=-1)
    xx.shape
    # viz(x)
    # dst3D = DST3D(N=n, J=2, L=[1, 2], B=[4, 8], norm=True, wave='db10')
    dst2D = DST2D(N=n, J=2, L=[1, 2], B=[4, 8], norm=True, wave='bior1.5')
    # dst3D = DST3D(N=n, J=2, L=[1, 2], B=[4, 8], norm=True, wave='rbio1.5')
    dst2D.forward(xx).shape
    xxrec = dst2D.inverse(dst2D(xx))
    xxrec.dtype, xxrec.shape
    print(f"\nReconstruction error: {tf.reduce_sum(tf.abs(xxrec - tf.cast(xx,dtype=tf.float64)))}\n")

    ## Example 2: model summary
    c = 2
    input_shape = (n,n,c)
    inputs = tf.keras.Input(input_shape)
    H  = DST2D(N=n, J=2, L=[1, 2], B=[4, 8], norm=True, wave='bior1.5')
    Hr = DST2D(N=n, J=2, L=[1, 2], B=[4, 8], norm=True, wave='bior1.5', transform='inverse')
    q = H(inputs)
    outputs = Hr(q)
    # outputs = q
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.summary()


Reconstruction error: 1.7217734451825677e-09



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dst2d_1 (DST2D)                 │ (None, 2, 13, 64, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ IDST3D (DST2D)                  │ (None, 64, 64, 2)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)